# 논문 그림 생성 노트북 — ResNet50 폐암 CT Block Encryption·Gaussian Blur 논문용

이 노트북은 논문 본문의 **표 1 · 표 2 · 표 4 · 표 7**을 `그림_1_블러.png` / `그림_2_블록.png` 스타일
(흰 배경, 제목 없음, 값별 컬러 막대, 최고/최저 기준선, 굵은 데이터 라벨)의 **깨끗한 흰 배경 그래프**로
변환합니다.

> ⚠️ 참고: 사용자가 예시로 준 3장의 이미지(그림_1_블러.png, 그림_2_블록.png, 그림_3_블록.png)는 실제로는
> **1차 논문(EfficientNetV2-S, 참고문헌 [2]) 결과**를 그린 그래프였습니다 (예: 그림_2의 "64×64: 86.7%",
> "8×8: 73.3%", "1×1: 52.9%" 값은 현재 논문 표 8의 EfficientNetV2-S 열과 정확히 일치, 현재 논문의
> ResNet50 표 1과는 다른 값입니다). 참고문헌 [2]와 표 8을 삭제하기로 하셨으므로, 이 노트북은 그
> **시각 스타일만 그대로 재현**하고 **데이터는 현재 논문(ResNet50, 표 1·2·4·7)의 실제 수치**로 교체했습니다.

## 실행 방법
1. 아래 "데이터 로드" 셀은 다음 3개 파일이 있으면 **자동으로 실제 실험 결과**를 읽어옵니다(4개 실험 노트북이 만드는 파일과 동일 경로):
   - `/workspace/all_block_results_resnet50_unified_v1.json`  (block_encryption 노트북 결과)
   - `/workspace/all_blur_results_resnet50_unified_v1.json`   (blur 노트북 결과)
   - `privacy_score_outputs/privacy_scores.csv`  (privacy_score 노트북 결과, 실행 위치 기준 상대경로)
2. 위 파일이 없으면 **논문에 이미 확정되어 실린 표 1·2·4·7의 수치**를 그대로 사용해 그래프를 그립니다(폴백).
   즉, 파일 유무와 관계없이 항상 논문에 실제로 실려야 할 값으로 그래프가 나옵니다.
3. 맨 아래 셀까지 순서대로 실행하면 `paper_figures/` 폴더에 PNG(300dpi)가 저장됩니다.


## 1. 라이브러리 & 한글 폰트 설정 (흰 배경 테마)

In [203]:
import os, json, glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ── Windows 로컬 NanumGothic 폰트 ─────────────────────────────
font_path = r"fonts\NanumGothic.ttf"
font_bold_path = r"fonts\NanumGothicBold.ttf"

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    fm.fontManager.addfont(font_bold_path)

    font_name = fm.FontProperties(fname=font_path).get_name()

    print(f"한글 폰트 적용: {font_name}")
    print(f"일반: {font_path}")
    print(f"Bold: {font_bold_path}")
else:
    font_name = "DejaVu Sans"
    print("나눔고딕 폰트를 찾지 못했습니다.")

# ── 흰 배경(라이트) 테마 ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [font_name, "DejaVu Sans"],
    "axes.unicode_minus": False,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "axes.edgecolor": "#222222",
    "axes.labelcolor": "#222222",

    "xtick.color": "#222222",
    "ytick.color": "#222222",
    "text.color": "#222222",

    "grid.color": "#d9d9d9",
    "grid.linestyle": "-",
    "grid.alpha": 0.8,

    "legend.facecolor": "white",
    "legend.edgecolor": "#bbbbbb",
    "legend.labelcolor": "#222222",
})

OUT_DIR = "paper_figures"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"출력 폴더: {os.path.abspath(OUT_DIR)}")

한글 폰트 적용: NanumGothic
일반: fonts\NanumGothic.ttf
Bold: fonts\NanumGothicBold.ttf
출력 폴더: .\2026\폐암 논문\폐암 2차 논문\코드\paper_figures


## 2. 공통 스타일 함수

`그림_1_블러.png` / `그림_2_블록.png`와 동일한 시각 문법:
- 제목 없음(캡션은 한글 워드/한글 프로그램에서 "그림 N. ..." 형태로 표 밖에 직접 붙이는 것을 권장)
- 값 크기에 따라 초록(높음) → 노랑 → 주황(낮음)으로 이어지는 컬러 막대 (RdYlGn 계열, **값 기준**으로 통일 —
  원본 예시 이미지 2장은 색 방향이 서로 반대였는데, 이 노트북은 "높을수록 초록"으로 통일했습니다)
- 막대 위 굵은 값 라벨
- 최고/최저 기준선(점선) + 필요 시 기준선(예: 80%) 을 범례로 표시


In [204]:
import matplotlib.colors as mcolors

def _bar_colors(values, vmin=None, vmax=None, cmap_name="RdYlGn"):
    """값이 높을수록 초록, 낮을수록 빨강/주황이 되도록 색을 매긴다."""
    vals = np.asarray(values, dtype=float)

    lo = vmin if vmin is not None else np.nanmin(vals)
    hi = vmax if vmax is not None else np.nanmax(vals)

    if hi - lo < 1e-9:
        norm_vals = np.ones_like(vals) * 0.6
    else:
        norm_vals = (vals - lo) / (hi - lo)
        norm_vals = 0.15 + norm_vals * 0.75

    cmap = plt.get_cmap(cmap_name)
    return [cmap(v) for v in norm_vals]


def plot_clean_bar(
    labels, values, xlabel, ylabel, save_name,
    value_fmt="{:.1f}%", ymax=None, ymin=0,
    show_max=True, show_min=True,
    threshold=None, threshold_label=None,
    figsize=(11, 5.6), rotation=0,
    value_fontsize=17,
):
    """막대 그래프 생성 함수."""

    values = np.asarray(values, dtype=float)
    colors = _bar_colors(values)

    fig, ax = plt.subplots(figsize=figsize)

    bars = ax.bar(
        labels,
        values,
        color=colors,
        edgecolor="none",
        zorder=3,
        width=0.62
    )

    # 막대 위 값 라벨
    pad = (np.nanmax(values) - np.nanmin(values)) * 0.02 + 0.4

    for b, v in zip(bars, values):
        ax.text(
            b.get_x() + b.get_width() / 2,
            v + pad,
            value_fmt.format(v),
            ha="center",
            va="bottom",
            fontsize=value_fontsize,
            fontweight="bold",
            color="#1a1a1a"
        )

    # 기준선
    handles, labels_leg = [], []

    if show_max:
        mx = np.nanmax(values)
        h = ax.axhline(
            mx,
            color="#e05a5a",
            linestyle="--",
            linewidth=1.4,
            zorder=2
        )
        handles.append(h)
        labels_leg.append(
            f"Max {mx:.1f}{'%' if '%' in value_fmt else ''}"
        )

    if show_min:
        mn = np.nanmin(values)
        h = ax.axhline(
            mn,
            color="#2f9e8f",
            linestyle="--",
            linewidth=1.4,
            zorder=2
        )
        handles.append(h)
        labels_leg.append(
            f"Min {mn:.1f}{'%' if '%' in value_fmt else ''}"
        )

    if threshold is not None:
        h = ax.axhline(
            threshold,
            color="#888888",
            linestyle=":",
            linewidth=1.4,
            zorder=2
        )
        handles.append(h)
        labels_leg.append(
            threshold_label or f"{threshold} Threshold"
        )

    # 축 제목
    ax.set_xlabel(
        xlabel,
        fontsize=25,
        fontweight="bold"
    )

    ax.set_ylabel(
        ylabel,
        fontsize=25,
        fontweight="bold"
    )

    # Y축 범위
    top = ymax if ymax is not None else np.nanmax(values) * 1.18
    ax.set_ylim(ymin, top)

    # Grid
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)

    # 눈금
    ax.tick_params(
        axis="x",
        labelrotation=rotation,
        labelsize=14
    )

    ax.tick_params(
        axis="y",
        labelsize=14
    )

    for lbl in ax.get_xticklabels():
        lbl.set_fontweight("bold")

    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    # 테두리
    for spine in ax.spines.values():
        spine.set_color("#222222")
        spine.set_linewidth(0.9)

    # 범례
    if handles:
        leg = ax.legend(
            handles,
            labels_leg,
            loc="upper right",
            frameon=True,
            fontsize=19
        )

        for text in leg.get_texts():
            text.set_fontweight("bold")

    # 저장
    fig.tight_layout()

    out_path = os.path.join(OUT_DIR, save_name)

    fig.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print(f"저장 완료: {out_path}")

    return out_path

## 3. 데이터 로드 (표 1 · 표 2) — 실제 실험 결과 우선, 없으면 논문 확정치 폴백

`all_results` 딕셔너리 구조는 4개 실험 노트북(`block_encryption_resnet50_rebuilt_KO.ipynb`,
`blur_resnet50_rebuilt2_KO.ipynb`)의 Cell 12(전체 실험 루프)가 만드는 것과 동일합니다
(`acc`, `roc_auc`, `per_class_acc` 키).

In [205]:
BLOCK_SIZES = [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
BLUR_SIGMAS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

# 논문 표 1 / 표 2에 실제로 게재된 확정 수치 (실험 파일이 없을 때의 폴백 데이터)
TABLE1_BLOCK_FALLBACK = {   # block_size: (accuracy%, roc_auc)
    512: (98.17, 0.9824), 256: (97.25, 0.9950), 128: (97.25, 0.9922),
    64:  (95.41, 0.9893), 32:  (88.99, 0.9634), 16:  (80.73, 0.9366),
    8:   (79.82, 0.9244), 4:   (61.47, 0.8259), 2:   (63.30, 0.7900),
    1:   (45.87, 0.6808),
}
TABLE2_BLUR_FALLBACK = {   # sigma: (accuracy%, roc_auc)
    0: (98.17, 0.9824), 1: (93.58, 0.9872), 2: (82.57, 0.9765),
    4: (85.32, 0.9657), 8: (80.73, 0.8825), 16: (66.97, 0.8376),
    32: (59.63, 0.6908), 64: (51.38, 0.6168), 128: (51.38, 0.6608),
    256: (51.38, 0.4997),
}

def _try_load_json(*candidate_paths):
    for p in candidate_paths:
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                return json.load(f), p
    return None, None


# ── 표 1: Block Encryption 단독 ───────────────────────────────────────
_raw, _src = _try_load_json(
    "/workspace/all_block_results_resnet50_unified_v1.json",
    "all_block_results_resnet50_unified_v1.json",
)
if _raw:
    block_acc = {int(k): v["acc"] * 100 for k, v in _raw.items()}
    print(f"[표 1] 실제 실험 결과 로드: {_src}  ({len(block_acc)}개 블록 크기)")
else:
    block_acc = {k: v[0] for k, v in TABLE1_BLOCK_FALLBACK.items()}
    print("[표 1] 실험 결과 파일 없음 → 논문 확정 수치(표 1)로 대체")

# ── 표 2: Gaussian Blur 단독 ──────────────────────────────────────────
_raw, _src = _try_load_json(
    "/workspace/all_blur_results_resnet50_unified_v1.json",
    "all_blur_results_resnet50_unified_v1.json",
)
if _raw:
    blur_acc = {int(k): v["acc"] * 100 for k, v in _raw.items()}
    print(f"[표 2] 실제 실험 결과 로드: {_src}  ({len(blur_acc)}개 시그마)")
else:
    blur_acc = {k: v[0] for k, v in TABLE2_BLUR_FALLBACK.items()}
    print("[표 2] 실험 결과 파일 없음 → 논문 확정 수치(표 2)로 대체")


[표 1] 실험 결과 파일 없음 → 논문 확정 수치(표 1)로 대체
[표 2] 실험 결과 파일 없음 → 논문 확정 수치(표 2)로 대체


## 4. 그림 — 표 1 대체: Block Encryption 단독, 블록 크기별 정확도

In [206]:
labels_block = ["Original" if bs == 512 else ("1×1\npx" if bs == 1 else str(bs)) for bs in BLOCK_SIZES]
values_block = [block_acc[bs] for bs in BLOCK_SIZES]

plot_clean_bar(
    labels_block, values_block,
    xlabel="Block Size (pixels)", ylabel="Accuracy (%)",
    save_name="fig_table1_block_accuracy.png",
    value_fmt="{:.2f}%", ymax=110,
    threshold=80, threshold_label="80% Threshold",
)


저장 완료: paper_figures\fig_table1_block_accuracy.png


'paper_figures\\fig_table1_block_accuracy.png'

## 5. 그림 — 표 2 대체: Gaussian Blur 단독, σ별 정확도

In [207]:
labels_blur = [f"σ={s}" for s in BLUR_SIGMAS]
values_blur = [blur_acc[s] for s in BLUR_SIGMAS]

plot_clean_bar(
    labels_blur, values_blur,
    xlabel="Blur Sigma (σ)", ylabel="Accuracy (%)",
    save_name="fig_table2_blur_accuracy.png",
    value_fmt="{:.2f}%", ymax=110,
    threshold=80, threshold_label="80% Threshold",
)


저장 완료: paper_figures\fig_table2_blur_accuracy.png


'paper_figures\\fig_table2_blur_accuracy.png'

## 6. 데이터 로드 (표 4 · Privacy Score) — 실제 실험 결과 우선, 없으면 논문 확정치 폴백

`privacy_score_ssim_psnr_updated.ipynb`가 만드는 `privacy_scores.csv`
(열: `type`, `block_size`, `sigma`, `privacy_score`)를 우선 사용합니다.

In [208]:
import csv

# 논문 표 4에 실제로 게재된 확정 Privacy Score (실험 파일이 없을 때의 폴백 데이터)
TABLE4_BLOCK_SCORE_FALLBACK = {
    512: 0.00, 256: 49.65, 128: 45.99, 64: 47.70, 32: 47.40,
    16: 49.20, 8: 49.57, 4: 49.27, 2: 48.82, 1: 47.14,
}
TABLE4_BLUR_SCORE_FALLBACK = {
    0: 0.00, 1: 0.75, 2: 2.48, 4: 6.10, 8: 11.45,
    16: 17.70, 32: 24.75, 64: 30.33, 128: 32.36, 256: 34.62,
}

def _try_load_privacy_csv(*candidate_paths):
    for p in candidate_paths:
        if os.path.exists(p):
            rows = []
            with open(p, encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                rows = list(reader)
            return rows, p
    return None, None

_rows, _src = _try_load_privacy_csv(
    "privacy_score_outputs/privacy_scores.csv",
    "/workspace/privacy_score_outputs/privacy_scores.csv",
    "privacy_scores.csv",
)

if _rows:
    block_score = {}
    blur_score = {}
    for r in _rows:
        if r.get("type") == "block" and r.get("block_size"):
            block_score[int(float(r["block_size"]))] = float(r["privacy_score"])
        elif r.get("type") == "blur" and r.get("sigma"):
            blur_score[int(float(r["sigma"]))] = float(r["privacy_score"])
    # 원본(0점) 행 보정: block=512, sigma=0
    block_score.setdefault(512, 0.0)
    blur_score.setdefault(0, 0.0)
    print(f"[표 4] 실제 실험 결과 로드: {_src}")
else:
    block_score = dict(TABLE4_BLOCK_SCORE_FALLBACK)
    blur_score = dict(TABLE4_BLUR_SCORE_FALLBACK)
    print("[표 4] 실험 결과 파일 없음 → 논문 확정 수치(표 4)로 대체")


[표 4] 실험 결과 파일 없음 → 논문 확정 수치(표 4)로 대체


## 7. 그림 — 표 4 대체 ①: Block Encryption 단독, 블록 크기별 Privacy Score

In [209]:
values_block_score = [block_score[bs] for bs in BLOCK_SIZES]

plot_clean_bar(
    labels_block, values_block_score,
    xlabel="Block Size (pixels)", ylabel="Privacy Score (0-100)",
    save_name="fig_table4_block_privacy_score.png",
    value_fmt="{:.2f}", ymax=58,
    threshold=50, threshold_label="Theoretical Max: 50",
    value_fontsize=20,
)


저장 완료: paper_figures\fig_table4_block_privacy_score.png


'paper_figures\\fig_table4_block_privacy_score.png'

## 8. 그림 — 표 4 대체 ②: Gaussian Blur 단독, σ별 Privacy Score

In [215]:
values_blur_score = [blur_score[s] for s in BLUR_SIGMAS]

plot_clean_bar(
    labels_blur, values_blur_score,
    xlabel="Blur Sigma (σ)", ylabel="Privacy Score (0-100)",
    save_name="fig_table4_blur_privacy_score.png",
    value_fmt="{:.2f}", ymax=58,
    threshold=50, threshold_label="Theoretical Max: 50",
    value_fontsize=25,
)


저장 완료: paper_figures\fig_table4_blur_privacy_score.png


'paper_figures\\fig_table4_blur_privacy_score.png'

## 9. 그림 — 표 7 대체: Privacy Score 구간별 최고 Accuracy

표 7은 6개 구간뿐이라 막대그래프로 바꿔도 정보 손실이 없습니다. 막대 위에는 정확도뿐 아니라
해당 구간의 **최고 조건 이름**도 함께 표기해, 표에 있던 정보를 그대로 보존합니다.

In [211]:
privacy_bins = ["0~5", "5~15", "15~25", "30~35", "35~45", "45~50"]
bin_best_acc  = [98.17, 87.16, 66.97, 51.38, 57.80, 98.17]
bin_condition = ["Original", "Block512+σ4", "Block512+σ16",
                  "Performance\nCollapse Zone", "Block64+σ256", "Block256+σ0"]

fig, ax = plt.subplots(figsize=(11, 6))
colors = _bar_colors(bin_best_acc)
bars = ax.bar(privacy_bins, bin_best_acc, color=colors, edgecolor="none", zorder=3, width=0.6)

for b, v, cond in zip(bars, bin_best_acc, bin_condition):
    ax.text(b.get_x() + b.get_width() / 2, v + 1.5, f"{v:.2f}%",
            ha="center", va="bottom", fontsize=25, fontweight="bold", color="#1a1a1a")
    ax.text(
        b.get_x() + b.get_width() / 2,
        4,
        cond,
        ha="center",
        va="bottom",
        fontsize=20,
        fontweight="bold",
        color="#000000",
        rotation=90
    )

ax.set_xlabel("Privacy Score Range", fontsize=25, fontweight="bold")
ax.set_ylabel("Best Accuracy in Range (%)", fontsize=25, fontweight="bold")
ax.set_ylim(0, 112)
ax.grid(axis="y", zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelsize=25)
ax.tick_params(axis="y", labelsize=25)
for lbl in ax.get_xticklabels():
    lbl.set_fontweight("bold")
for lbl in ax.get_yticklabels():
    lbl.set_fontweight("bold")
for spine in ax.spines.values():
    spine.set_color("#222222"); spine.set_linewidth(0.9)

fig.tight_layout()
out_path = os.path.join(OUT_DIR, "fig_table7_privacyscore_bins_best_accuracy.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"저장 완료: {out_path}")


저장 완료: paper_figures\fig_table7_privacyscore_bins_best_accuracy.png


## 10. (선택/보너스) 블록 크기 → "암호화 강도" 산점도

사용자가 예시로 준 `그림_3_블록.png`(1차 논문 데이터)를 역산해 보면, x축 "암호화 강도"는
`100 × (1 − log2(block_size)/log2(512))` 로 정의된 값이었습니다(원본=0, 1×1픽셀=100).
논문 본문에는 이런 지표가 정의되어 있지 않으므로, **이 그림을 논문에 넣으려면 3.2.1절이나 그림 설명에
이 수식을 먼저 정의해야 합니다.** 정의 없이 쓰면 심사자가 "암호화 강도"가 무엇인지 되물을 수 있습니다.
표 1을 이미 막대그래프로 옮겼으므로 **이 그림은 필수가 아니라 보너스**입니다.

In [212]:
def encryption_strength(block_size):
    return 100.0 * (1 - np.log2(block_size) / np.log2(512))

strengths = [encryption_strength(bs) for bs in BLOCK_SIZES]
accs = [block_acc[bs] for bs in BLOCK_SIZES]
point_labels = ["Original" if bs == 512 else (f"{bs}×{bs}" if bs > 1 else "1×1") for bs in BLOCK_SIZES]

fig, ax = plt.subplots(figsize=(13, 8.5))
sc = ax.scatter(strengths, accs, c=accs, cmap="RdYlGn", s=1500,
                 edgecolors="white", linewidths=2, vmin=min(accs), vmax=max(accs), zorder=3)
for i, (x, y, lbl) in enumerate(zip(strengths, accs, point_labels)):
    if i % 2 == 0:
        ax.annotate(lbl, (x, y), xytext=(0, 34), textcoords="offset points",
                    fontsize=21, fontweight="bold", ha="center", va="bottom")
    else:
        ax.annotate(lbl, (x, y), xytext=(0, -34), textcoords="offset points",
                    fontsize=21, fontweight="bold", ha="center", va="top")

ax.set_xlim(-8, 108)
ax.set_ylim(22, 112)
ax.set_xlabel("Encryption Strength (100×(1−log2(block_size)/log2(512)))", fontsize=18, fontweight="bold")
ax.set_ylabel("Accuracy (%)", fontsize=18, fontweight="bold")
ax.grid(True, zorder=0)
ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=14)
for lbl in ax.get_xticklabels():
    lbl.set_fontweight("bold")
for lbl in ax.get_yticklabels():
    lbl.set_fontweight("bold")
for spine in ax.spines.values():
    spine.set_color("#222222"); spine.set_linewidth(0.9)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Accuracy (%)", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=13)
for lbl in cbar.ax.get_yticklabels():
    lbl.set_fontweight("bold")

fig.tight_layout()
out_path = os.path.join(OUT_DIR, "fig_bonus_block_encryption_strength_scatter.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"저장 완료: {out_path}")


workspace\AppData\Local\Temp\ipykernel_29024\3857756903.py:38: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) NanumGothic.
  fig.tight_layout()
workspace\AppData\Local\Temp\ipykernel_29024\3857756903.py:40: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) NanumGothic.
  fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")


저장 완료: paper_figures\fig_bonus_block_encryption_strength_scatter.png


## 11. 결과 요약

생성된 파일은 모두 `paper_figures/` 폴더에 저장됩니다.

| 파일명 | 대체 대상 | 논문 내 위치 |
|---|---|---|
| `fig_table1_block_accuracy.png` | 표 1 | 4.1절 |
| `fig_table2_blur_accuracy.png` | 표 2 | 4.2절 |
| `fig_table4_block_privacy_score.png` | 표 4 (블록 부분) | 4.4절 |
| `fig_table4_blur_privacy_score.png` | 표 4 (블러 부분) | 4.4절 |
| `fig_table7_privacyscore_bins_best_accuracy.png` | 표 7 | 5.1절 |
| `fig_bonus_block_encryption_strength_scatter.png` | (선택) | 사용 안 해도 무방 |

표 3 · 표 5(100개 조합 원자료 표)와 표 6(Pareto n=3), 표 8(1차 논문 비교표)에 대한 처리 방향은
별도로 드린 **수정사항 안내 문서(.docx)**의 "7. 표 → 그래프 전환 안내"를 참고하세요.
